In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from tqdm.auto import tqdm
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


/home/thomas/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test_df = pd.read_csv("../data/test.csv")

print(test_df.shape)
print(test_df.columns)
print(test_df.head())

(3, 4)
Index(['id', 'prompt', 'response_a', 'response_b'], dtype='str')
        id                                             prompt  \
0   136060  ["I have three oranges today, I ate an orange ...   
1   211333  ["You are a mediator in a heated political deb...   
2  1233961  ["How to initialize the classification head wh...   

                                          response_a  \
0                    ["You have two oranges today."]   
1  ["Thank you for sharing the details of the sit...   
2  ["When you want to initialize the classificati...   

                                          response_b  
0  ["You still have three oranges. Eating an oran...  
1  ["Mr Reddy and Ms Blue both have valid points ...  
2  ["To initialize the classification head when p...  


In [3]:
def build_reward_input(row, response_key):
    conversation = []

    for prompt, response_a, response_b in zip(
        row["prompt_parsed"],
        row["response_a_parsed"],
        row["response_b_parsed"]
    ):
        if prompt is not None:
            conversation.append(f"User: {prompt}")

        response = response_a if response_key == "a" else response_b

        if response is not None:
            conversation.append(f"Assistant: {response}")

    return "\n".join(conversation)

In [4]:
def tokenize_head_tail(
    text,
    tokenizer,
    max_length=512,
    head_ratio=0.5
):
    # Tokenize without special tokens
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    # Reserve space for [CLS] and [SEP]
    available_length = max_length - 2

    # Short sequence: keep everything
    if len(token_ids) <= available_length:

        input_ids = (
            [tokenizer.cls_token_id]
            + token_ids
            + [tokenizer.sep_token_id]
        )

    # Long sequence: keep head + tail
    else:

        head_length = int(
            available_length * head_ratio
        )

        tail_length = (
            available_length - head_length
        )

        head = token_ids[:head_length]
        tail = token_ids[-tail_length:]

        input_ids = (
            [tokenizer.cls_token_id]
            + head
            + tail
            + [tokenizer.sep_token_id]
        )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids)
    }

In [18]:
def tokenize_dataframe(df, tokenizer, max_length=512):
    enc_a = []
    enc_b = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Tokenizing"
    ):
        enc_a.append(
            tokenize_head_tail(
                row["input_a"],
                tokenizer,
                max_length
            )
        )

        enc_b.append(
            tokenize_head_tail(
                row["input_b"],
                tokenizer,
                max_length
            )
        )


    return enc_a, enc_b

In [19]:
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

MAX_LENGTH = 512

tokenizer.save_pretrained("./deberta-v3-small")
backbone.save_pretrained("./deberta-v3-small")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1199.37it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

In [20]:
class RewardModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone
        hidden_size = backbone.config.hidden_size

        self.attention_pool = nn.Linear(
            hidden_size,
            1
        )

        self.reward_head = nn.Linear(
            hidden_size,
            1
        )

        self.raw_tie_param = nn.Parameter(
            torch.tensor(0.5413)
        )

    def forward(self, input_ids, attention_mask):
        output = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden = output.last_hidden_state

        attention_scores = self.attention_pool(
            hidden
        ).squeeze(-1)

        attention_scores = attention_scores.masked_fill(
            attention_mask == 0,
            -1e9
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=1
        )

        pooled = torch.sum(
            hidden * attention_weights.unsqueeze(-1),
            dim=1
        )

        reward = self.reward_head(pooled)

        return reward.squeeze(-1)

    def get_tie_param(self):
        return F.softplus(self.raw_tie_param)

In [26]:
class PreferenceDataset(Dataset):

    def __init__(self, enc_a, enc_b):
        self.enc_a = enc_a
        self.enc_b = enc_b


    def __len__(self):
        return len(self.enc_a)

    def __getitem__(self, idx):
        return (
            self.enc_a[idx],
            self.enc_b[idx]
        )


In [27]:
def collate_fn(batch, tokenizer):
    batch_a = [item[0] for item in batch]
    batch_b = [item[1] for item in batch]


    enc_a = tokenizer.pad(
        batch_a,
        padding=True,
        return_tensors="pt"
    )

    enc_b = tokenizer.pad(
        batch_b,
        padding=True,
        return_tensors="pt"
    )

    enc_a.pop("token_type_ids", None)
    enc_b.pop("token_type_ids", None)

    return enc_a, enc_b

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

reward_model = RewardModel(backbone).to(device)

In [29]:
def preference_probabilities(reward_a, reward_b, tie_param=1.0):
    """
    Convert two reward scores into probabilities of:
    A wins, B wins, Tie
    """

    exp_a = torch.exp(reward_a)
    exp_b = torch.exp(reward_b)

    tie_term = (
        2 * tie_param *
        torch.exp((reward_a + reward_b) / 2)
    )

    denominator = exp_a + exp_b + tie_term

    prob_a = exp_a / denominator
    prob_b = exp_b / denominator
    prob_tie = tie_term / denominator

    return prob_a, prob_b, prob_tie

In [30]:
test_df["prompt_parsed"] = test_df["prompt"].apply(json.loads)
test_df["response_a_parsed"] = test_df["response_a"].apply(json.loads)
test_df["response_b_parsed"] = test_df["response_b"].apply(json.loads)

test_df["input_a"] = test_df.apply(
    lambda row: build_reward_input(row, "a"),
    axis=1
)

test_df["input_b"] = test_df.apply(
    lambda row: build_reward_input(row, "b"),
    axis=1
)

test_dataset = PreferenceDataset(
    *tokenize_dataframe(
        test_df,
        tokenizer,
        MAX_LENGTH
    )
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda batch: collate_fn(batch, tokenizer)
)

print()

Tokenizing: 100%|██████████| 3/3 [00:00<00:00, 1041.63it/s]

In [31]:
checkpoint = torch.load(
    "../models/epoch_3.pt",
    map_location=device
)

reward_model.load_state_dict(
    checkpoint["model_state_dict"]
)

reward_model.to(device)
reward_model.eval()

print()

In [ ]:
all_probabilities = []

with torch.no_grad():

    for enc_a, enc_b in tqdm(test_loader, desc="Inference"):

        enc_a = {
            k: v.to(device)
            for k, v in enc_a.items()
        }

        enc_b = {
            k: v.to(device)
            for k, v in enc_b.items()
        }

        reward_a = reward_model(**enc_a)
        reward_b = reward_model(**enc_b)

        prob_a, prob_b, prob_tie = preference_probabilities(
            reward_a,
            reward_b,
            tie_param=1.0
        )

        logits = torch.stack(
            [prob_a, prob_b, prob_tie],
            dim=1
        )

        probabilities = F.softmax(logits, dim=1)

        all_probabilities.append(
            probabilities.cpu()
        )

all_probabilities = torch.cat(
    all_probabilities,
    dim=0
)

print(all_probabilities.shape)

Inference: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]

torch.Size([3, 3])


In [33]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": all_probabilities[:, 0].numpy(),
    "winner_model_b": all_probabilities[:, 1].numpy(),
    "winner_tie": all_probabilities[:, 2].numpy(),
})

In [34]:
print(
    submission[
        ["winner_model_a", "winner_model_b", "winner_tie"]
    ].sum(axis=1).head()
)

0    1.0
1    1.0
2    1.0
dtype: float32


In [35]:
submission.to_csv("submission.csv", index=False)


In [1]:
import kagglehub

# Replace with path to directory containing model files.
LOCAL_MODEL_DIR = '../models/pairwise_model_v3/epoch_2.pt'

MODEL_SLUG = 'deberta_classification_v3_epoch2' # Replace with model slug.

# Learn more about naming model variations at
# https://www.kaggle.com/docs/models#name-model.
VARIATION_SLUG = 'default' # Replace with variation slug.

kagglehub.model_upload(
  handle = f"xiethomas/{MODEL_SLUG}/pyTorch/{VARIATION_SLUG}",
  local_model_dir = LOCAL_MODEL_DIR,
  version_notes = 'Update 2026-09-08')

Uploading Model https://kaggle.com/models/xiethomas/deberta_classification_v3_epoch2/pyTorch/default ...


/home/thomas/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model 'deberta_classification_v3_epoch2' does not exist or access is forbidden for user 'xiethomas'. Creating or handling Model...
Model 'deberta_classification_v3_epoch2' Created.
Starting upload for file ../models/pairwise_model_v3/epoch_2.pt


Uploading: 100%|██████████| 1.72G/1.72G [02:47<00:00, 10.3MB/s] 

Upload successful: ../models/pairwise_model_v3/epoch_2.pt (2GB)


Your model instance has been created.
Files are being processed...
See at: https://kaggle.com/models/xiethomas/deberta_classification_v3_epoch2/pyTorch/default


In [8]:
import kagglehub

handle = 'xiethomas/deberta'
local_dataset_dir = 'deberta-v3-small'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset_dir)

# # You can then create a new version of this dataset and include version notes.
# kagglehub.dataset_upload(handle, local_dataset_dir, version_notes='improved data')

# # You can also specify a list of patterns for files/dirs to ignore.
# # These patterns are combined with 'kagglehub.datasets.DEFAULT_IGNORE_PATTERNS'
# # to determine which files and directories to exclude. 
# # To ignore entire directories, include a trailing slash (/) in the pattern.
# kagglehub.dataset_upload(handle, local_dataset_dir, ignore_patterns=["original/", "*.tmp"])


Uploading Dataset https://kaggle.com/datasets/xiethomas/deberta ...
Starting upload for file deberta-v3-small/model.safetensors


Uploading: 100%|██████████| 565M/565M [00:48<00:00, 11.6MB/s]  

Upload successful: deberta-v3-small/model.safetensors (539MB)
Starting upload for file deberta-v3-small/config.json



Uploading: 100%|██████████| 904/904 [00:00<00:00, 2.41kB/s]

Upload successful: deberta-v3-small/config.json (904B)
Starting upload for file deberta-v3-small/tokenizer.json



Uploading: 100%|██████████| 8.34M/8.34M [00:01<00:00, 7.28MB/s]

Upload successful: deberta-v3-small/tokenizer.json (8MB)
Starting upload for file deberta-v3-small/tokenizer_config.json



Uploading: 100%|██████████| 538/538 [00:00<00:00, 1.51kB/s]

Upload successful: deberta-v3-small/tokenizer_config.json (538B)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/xiethomas/deberta
